# Predicting F1 Pit Stops

In [ ]:
import os
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

## Data Read

In [ ]:
# Kaggle Notebook
# df_train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv')
# df_test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')

# local 
data_path = "/mnt/e/Kaggle_F1_Pit_Stops"
train_dp = os.path.join(data_path, "train.csv")
test_dp = os.path.join(data_path, "test.csv")

df_train = pd.read_csv(train_dp)
df_test = pd.read_csv(test_dp)

In [ ]:
df_train.head()

In [ ]:
df_train.describe()

In [ ]:
df_train.info()

## Data Analysis

In [ ]:
df_train['PitStop'].hist(bins=50)
plt.show()

In [ ]:
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

corr = df_train[numeric_cols].corr()

corr['PitStop'].sort_values(ascending=False)

In [ ]:
# for col in numeric_cols[:10]:
#     plt.figure(figsize=(6,3))
#     df_train[col].hist(alpha=0.5, label='train')
#     df_test[col].hist(alpha=0.5, label='test')
#     plt.legend()
#     plt.title(col)
#     plt.show()

## Dataset Preprocessing

In [ ]:
TARGET = 'PitNextLap'

dset_train = df_train.copy()
dset_test = df_test.copy()

X = dset_train.drop(columns=[TARGET, 'id'])
y = dset_train[TARGET]

X_test = dset_test.drop(columns=['id'])

In [ ]:
cat_cols = X.select_dtypes(include='object').columns

for col in cat_cols:

    le = LabelEncoder()

    full_data = pd.concat([
        X[col],
        X_test[col]
    ]).astype(str)

    le.fit(full_data)

    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

## Model Train and Test

In [ ]:
# training and test setup
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    # # Random Forest
    # model = RandomForestClassifier(
    #     n_estimators=500,
    #     max_depth=15,
    #     min_samples_leaf=5,
    #     n_jobs=-1,
    #     random_state=42,
    #     class_weight='balanced'
    # )

    # Extra Trees
    model = ExtraTreesClassifier(
        n_estimators=1000,
        max_depth=None,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=False,
        n_jobs=-1,
        random_state=42,
        class_weight='balanced'
    )


    # train
    model.fit(X_train, y_train)

    # valid
    valid_pred = model.predict_proba(X_valid)[:,1]
    oof[valid_idx] = valid_pred
    score = roc_auc_score(y_valid, valid_pred)
    print(f'Fold {fold + 1}: {score:.5f}')

    # test
    preds += model.predict_proba(X_test)[:,1] / 5

In [ ]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'PitNextLap': preds
})

preds_save_path = os.path.join(data_path, 'submission.csv')
submission.to_csv(preds_save_path, index=False)